# Treatment means and contrasts

**STA 509 2.0 Introduction to Design and Analysis of Experiments** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta509.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [What an omnibus test leaves unanswered](#theory-01)
- [Contrasts and uncertainty](#theory-02)
- [Pairwise intervals and familywise error](#theory-03)
- [Fisher LSD and Tukey methods](#theory-04)
- [Reading comparison output](#theory-05)
- [Residual adequacy](#theory-06)
- [Remedies and scope](#theory-07)

## Detailed theory · Treatment comparisons and model adequacy

<a id="theory-01"></a>

### What an omnibus test leaves unanswered
An ANOVA F assesses equality of all treatment means, not every specific difference. A planned contrast answers a predeclared scientific question; exploratory pairwise comparisons inspect a family of possible differences. Distinguish those goals before choosing a multiplicity method.

<a id="theory-02"></a>

### Contrasts and uncertainty
A contrast $C=\sum_i c_i\mu_i$ has $\sum c_i=0$. Estimate it by $\hat C=\sum c_i\bar y_i$. For independent CRD group means with a common MSE,
$$SE(\hat C)=\sqrt{MS_E\sum_i c_i^2/n_i}.$$
A single planned contrast uses a matching t reference with error df. A coefficient multiplying an average must be divided across its contributing treatments; compare a control with the average of two others using −1,.5,.5, not −1,1,1.

<a id="theory-03"></a>

### Pairwise intervals and familywise error
There are a(a−1)/2 pairs. Repeated unadjusted 5% comparisons do not give 5% error for the whole family. Bonferroni uses alpha/m per test for a family of m; its two-sided intervals use a 1−alpha/(2m) t quantile. It is general but can be conservative.

<a id="theory-04"></a>

### Fisher LSD and Tukey methods
LSD uses an ordinary pairwise t margin from pooled MSE and needs carefully stated protection/planning; an omnibus significance filter is not a universal multiplicity guarantee for every comparison family. Tukey HSD addresses all pairwise comparisons under the classical model. With equal replication n,
$$HSD=q_{1-\alpha;a,\nu}\sqrt{MS_E/n}.$$
For unequal sizes, Tukey–Kramer uses $q\sqrt{MS_E(1/n_i+1/n_j)/2}$. A q critical value is not a t critical value.

<a id="theory-05"></a>

### Reading comparison output
Interpret the signed mean difference, compatible confidence interval and adjusted p-value together. Shared grouping letters are summaries of a chosen multiple-comparison method, not proof that means in a letter group are equal. Specify the family, adjustment, confidence level and subtraction direction.

<a id="theory-06"></a>

### Residual adequacy
In a CRD $e_{ij}=y_{ij}-\bar y_i$; in an additive block model residuals remove both treatment and block means. Inspect residuals versus fitted values, treatment, run order and a Q–Q reference. Curvature, funnels, time patterns and unusual cases indicate different issues. A residual normality test with p>.05 does not demonstrate all model assumptions.

<a id="theory-07"></a>

### Remedies and scope
Unequal variance may require a justified transformation or a heteroscedastic method; dependence requires a model/design appropriate for its source. Transforming the response changes the scale of mean contrasts and back-transformed summaries. Document choices and practical effects with uncertainty instead of treating a software p-value as a complete conclusion.

## Concepts covered

**Course points:** Treatment means and contrasts; ANOVA table; Tukey multiple comparisons; residual checks; normality/common variance.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Planned contrast: coefficient, estimate and SE table

### Question and assumptions

Using the CRD example, compare treatment A with the average of B and C. Define $C=-\mu_A+\frac12\mu_B+\frac12\mu_C$. The fitted error mean square is 2 with 6 df.

### Procedure

1. Choose contrast coefficients before inspecting results and check that they sum to zero.
2. Calculate each coefficient × group mean contribution.
3. Calculate MSE×sum(c_i²/n_i) and take its square root.
4. Construct the single planned-contrast t interval; distinguish it from a family of exploratory comparisons.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Treatment | Coefficient c | Mean | n | c × mean | c²/n |
| --- | --- | --- | --- | --- | --- |
| A | -1 | 9 | 3 | -9 | 0.333333 |
| B | 0.5 | 12 | 3 | 6 | 0.0833333 |
| C | 0.5 | 14 | 3 | 7 | 0.0833333 |

| Quantity | Value |
| --- | --- |
| Sum of coefficients | 0 |
| Contrast estimate | 4 |
| SE | 1 |
| t | 4 |
| df | 6 |
| 95% lower bound | 1.55309 |
| 95% upper bound | 6.44691 |
| Single planned-contrast decision | Reject zero contrast |

| Comparison family | Count | Bonferroni per-comparison alpha |
| --- | --- | --- |
| All pairs among 3 treatments | 3 | 0.0166667 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


names=["A","B","C"]; means=[9,12,14]; n=[3,3,3]; coeff=[-1,.5,.5]; MSE=2; df=6
rows=[[g,c,xb,ni,c*xb,c*c/ni] for g,c,xb,ni in zip(names,coeff,means,n)]
show_table(["Treatment","Coefficient c","Mean","n","c × mean","c²/n"],rows)
estimate=sum(r[4] for r in rows); se=sqrt(MSE*sum(r[5] for r in rows)); t=estimate/se; critical=2.4469118511449692
show_table(["Quantity","Value"],[["Sum of coefficients",sum(coeff)],["Contrast estimate",estimate],["SE",se],["t",t],["df",df],["95% lower bound",estimate-critical*se],["95% upper bound",estimate+critical*se],["Single planned-contrast decision","Reject zero contrast" if abs(t)>critical else "Do not reject"]])
show_table(["Comparison family","Count","Bonferroni per-comparison alpha"],[["All pairs among 3 treatments",3,.05/3]])
assert sum(coeff)==0 and estimate==4 and se==1


### Decision and interpretation

The estimated average of B and C is 4 units above A. This interval is for one planned contrast. If selecting many comparisons after seeing results, use an appropriate familywise method and its corresponding critical values or adjusted p-values.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
groups=[rng.normal(mu,5,20) for mu in [50,53,61,62]]
print('means',[g.mean().round(2) for g in groups]); print(stats.tukey_hsd(*groups))
print('Levene',stats.levene(*groups)); e=np.concatenate([g-g.mean() for g in groups]); print('Shapiro diagnostic',stats.shapiro(e))
fig,ax=plt.subplots(1,2); ax[0].boxplot(groups); ax[1].scatter(np.repeat(range(4),20),e); ax[1].axhline(0,color='k',ls='--'); plt.show()

### Practice set · After the F
1. Why are unadjusted pairwise tests risky after examining many groups?
2. How many pairwise comparisons are there for k=5 treatments?
3. What does a funnel-shaped residual plot suggest?
4. Give one reason a transformation complicates interpretation.

<details><summary>Answer sketches</summary>
1. Familywise Type I error rises as the number of tests grows. 2. k(k−1)/2=10. 3. Nonconstant variance. 4. Effects are on transformed scale and must be translated back; transformed mean is not generally the original-scale mean.
</details>

### Debrief
A significant omnibus F does not say which means differ. Report adjusted follow-ups.

Record the formula, assumptions, and one follow-up question.